In [1]:
from platform import python_version
python_version()
import numpy as np
import torch
import numpy as np
from arsf_envi_reader import envi_header
import shutil
import os

import json
import math
import affine
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
from osgeo import gdal,ogr,osr

import numpy as np
# import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

import rasterio
from rasterio.windows import Window

from tqdm import tqdm
# import multiprocess as mp
from numpy import trapz

C:\Users\laral\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
torch.cuda.is_available()

True

In [3]:
get_ipython().run_line_magic('config', 'Completer.use_jedi = False')

# using AVIRIS-NG to calculate the srf

In [2]:
in_header = envi_header.find_hdr_file(r"D:\wenqu\aviris\site2a_aviris_ng_srf_data\ang20190704t193319rfl\ang20190704t193319_rfl_v2v2_img.hdr")
header_data = envi_header.read_hdr_file(in_header)

In [3]:
# Get wavelengths and convert to NumPy array
low_res_wavelengths = header_data['wavelength'].split(',')
low_res_wavelengths = [float(w) for w in low_res_wavelengths]
low_res_wavelengths = np.array(low_res_wavelengths)


low_res_fwhm = header_data['fwhm'].split(',')
low_res_fwhm = [float(w) for w in low_res_fwhm]
low_res_fwhm = np.array(low_res_fwhm)

In [4]:
def gauss(x, f, w):
    sigma = f/(2 * np.sqrt(2 *np.log(2)))
    y = np.exp((-(x-w)**2) / (2 * sigma**2))
    return y

In [2]:
high_res_img = gdal.Open(r'E:\wenqu\2024_data\site7\site7_1')
high_res_reflectance = gdal.Open(r'E:\wenqu\2024_data\site7\site7_1').ReadAsArray() 

high_res_reflectance.shape

(273, 16826, 1340)

In [6]:
high_res_wavelength = [float(b.split(" ")[0]) for b in high_res_img.GetMetadata().values() if b != "nm"]
high_res_wavelength.sort()
high_res_wavelength = np.array(high_res_wavelength)

In [7]:
gaussian_weights = []
for n in range(len(low_res_fwhm)):
#     print(n)
    a = []
    for i in high_res_wavelength:
        x = gauss(i, low_res_fwhm[n], low_res_wavelengths[n])
        if x < 0.5:
            x = 0
            a.append(x)
        else:
            a.append(x)
#     print(a)
    gaussian_weights.append(a)

In [8]:
gaussian_weights = np.array(gaussian_weights)
print(gaussian_weights.shape)

(425, 273)


In [3]:
bands, H, W = high_res_reflectance.shape
N = H * W
high_res_img_flatten = high_res_reflectance.reshape(bands, -1).T   

In [10]:
high_res_img_flatten.shape

(14659920, 273)

In [11]:
srf = gaussian_weights                             # (nb, 273)
wl  = high_res_wavelength   
denoms = np.trapz(srf, wl, axis=1) + 1e-12   # (273,)

In [12]:
srf = gaussian_weights                             # (nb, 273)
wl  = high_res_wavelength   
denoms = np.trapz(srf, wl, axis=1) + 1e-12   # (273,)

device = torch.device("cuda")

gaussian_weights_tensor = torch.from_numpy(
    np.array(gaussian_weights).astype(np.float32)
).to(device)                                 # (425, 273)

high_res_img_flatten_tensor = torch.from_numpy(
    high_res_img_flatten.astype(np.float32)
).to(device)                                 # (1758084, 273)

denoms_tensor = torch.from_numpy(
    denoms.astype(np.float32)
).to(device)                                 # (425,)

wl_tensor = torch.from_numpy(
    high_res_wavelength.astype(np.float32)
).to(device)                                 # (273,)

N_pixels = high_res_img_flatten_tensor.shape[0]   # 注意这里是 0 维

batch_pixels = []
out_dir = "site7_3"
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)
os.makedirs(out_dir)

save_every = 1_000_000

with torch.no_grad():
    for i in tqdm(range(N_pixels)):
        # 这一行是第 i 个像素的 273 band
        pixel_spec = high_res_img_flatten_tensor[i, :]         # (273,)

        # 425 条 SRF × 这个像素的光谱 → (425, 273)
        nomin_inputs = gaussian_weights_tensor * pixel_spec    # 自动广播

        # 对波长积分：每行一条 SRF → (425,)
        nomins = torch.trapz(nomin_inputs, wl_tensor, dim=1)   # (425,)

        # 归一化
        pixel_single = nomins / denoms_tensor                  # (425,)

        batch_pixels.append(pixel_single.cpu().numpy())

        if (i + 1) % save_every == 0:
            arr = np.array(batch_pixels, dtype=np.float32)
            np.save(f"{out_dir}/{i:010d}.npy", arr)
            batch_pixels = []

# 最后一批
if len(batch_pixels):
    arr = np.array(batch_pixels, dtype=np.float32)
    np.save(f"{out_dir}/{N_pixels-1:010d}.npy", arr)
    batch_pixels = []


100%|██████████████████████████████████████████████████████████████████| 14659920/14659920 [1:07:40<00:00, 3610.70it/s]


In [13]:
print(os.getcwd())

C:\Users\laral\OneDrive\Documents\GitHub\hytools


# reshape the convoluted back to images

In [4]:
import os
import glob
import numpy as np

pixel_dir = r"E:\wenqu\2024_data\site7\numpy\site7_1"

files = sorted(glob.glob(os.path.join(pixel_dir, "*.npy")))
print(files)

pixel_all = np.vstack([np.load(f) for f in files])
 
print("pixel_all:", pixel_all.shape)
# (N_pixels, nb)


['E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0000999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0001999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0002999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0003999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0004999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0005999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0006999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0007999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0008999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0009999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0010999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0011999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0012999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0013999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\0014999999.npy', 'E:\\wenqu\\2024_data\\site7\\numpy\\site7_1\\00159999

In [5]:
nb = pixel_all.shape[1]        # 卷积后的波段数




In [6]:
# (N_pixels, nb) → (H, W, nb)
img_HWC = pixel_all.reshape(H, W, nb)
print("img_HWC shape:", img_HWC.shape)   # (H, W, nb)

img_HWC shape: (16826, 1340, 425)


# filter out non bands

In [7]:
band_max = np.max(np.abs(img_HWC), axis=(0, 1)) 
eps = 1e-6
valid_band_mask = band_max > eps

print("Total bands:", img_HWC.shape[2])
print("Valid bands:", np.sum(valid_band_mask))


Total bands: 425
Valid bands: 122


In [8]:
img_HWC_filtered = img_HWC[:, :, valid_band_mask]
img_HWC_filtered.shape

(16826, 1340, 122)

In [9]:
H, W, nb_valid = img_HWC_filtered.shape
src_path =r'E:\wenqu\2024_data\site7\site7_1'
src_ds = gdal.Open(src_path)

geotransform = src_ds.GetGeoTransform()
projection   = src_ds.GetProjection()
src_ds = None  # close

# -------------------------------------------------
# 3. Create a new GeoTIFF and write bands
# -------------------------------------------------
out_path = r"E:\wenqu\2024_data\site7\site7_ortho1.tif"

driver = gdal.GetDriverByName("GTiff")
# Create: cols, rows, bands, data type
out_ds = driver.Create(out_path, W, H, nb_valid, gdal.GDT_Float32)

# Set spatial reference
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)

# Write each band (GDAL uses band index starting at 1)
for b in range(nb_valid):
    band_array = img_HWC_filtered[:, :, b]    # (H, W)
    out_ds.GetRasterBand(b + 1).WriteArray(band_array)
    # Optional: set NoData
    # out_ds.GetRasterBand(b + 1).SetNoDataValue(-9999)

out_ds.FlushCache()
out_ds = None  # close and save

print("Saved filtered convolved image to:")
print(out_path)

Saved filtered convolved image to:
E:\wenqu\2024_data\site7\site7_ortho1.tif
